# Домашнее задание №5

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import load_breast_cancer, load_iris, load_wine
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.tree import DecisionTreeClassifier
import matplotlib.pyplot as plt

In [ ]:
np.random.seed(42)

## Задание 1: Математика критериев (Gini vs Entropy)

`Описание:` Сравните два основных критерия разбиения. Нужно реализовать расчеты вручную, чтобы понять, какой из них сильнее штрафует за «примеси».

`Что нужно сделать:`
1. Напишите функции для расчета **Gini Impurity** и **Entropy**.
2. Рассчитайте **Information Gain** для разбиения узла S [50+, 50-] на два узла: L [45+, 5-] и R [5+, 45-] по обоим критериям.

Напоминаем, что для некоторого узла $U$ формулу критериев выглядят так:

`Entropy`
$$\mathbb{H}(U) =  -\sum_{i}p_i \ log(p_i)$$

`Gini`
$$ \mathbb{H}(U) = 1 -\sum_{i}p_i^2$$

**Запрещено использовать sklearn для расчёта Gini / Entropy.**

In [ ]:
def get_gini(labels):
    # labels: список или массив меток классов
    pass

def get_entropy(labels):
    # labels: список или массив меток классов
    pass

# Исходные данные для расчета IG
parent = [1]*50 + [0]*50
left = [1]*45 + [0]*5
right = [1]*5 + [0]*45

ig_gini = # Ваш расчет
ig_entropy = # Ваш расчет

In [ ]:
# Базовые проверки
assert np.isclose(get_gini(parent), 0.5, atol=1e-3)
assert np.isclose(get_entropy(parent), 1.0, atol=1e-3)

# Проверка IG
assert np.isclose(ig_gini, 0.4, atol=1e-3)
assert np.isclose(ig_entropy, 0.714, atol=1e-3)

# Дополнительные проверки (анти-чит)
y1 = [0, 0, 1, 1]
assert abs(get_gini(y1) - 0.5) < 1e-6
assert abs(get_entropy(y1) - 1.0) < 1e-6

# Чистый узел
y2 = [1,1,1,1]
assert get_gini(y2) == 0
assert get_entropy(y2) == 0

# Несбалансированный случай
y3 = [1]*9 + [0]
assert get_gini(y3) < 0.3
assert get_entropy(y3) < 0.5

## Задание 2: Оптимальная обрезка (CCP Pruning)

`Описание:` Использование метода «Cost-Complexity Pruning» для автоматического поиска лучшей глубины.

`Что нужно сделать:`
1. Получите путь эффективных `ccp_alphas` для дерева на `load_iris`.
2. Найдите такое `alpha` (из полученного списка), которое максимизирует средний `cross_val_score`.

In [ ]:
iris = load_iris()
X, y = iris.data, iris.target
dt_full = DecisionTreeClassifier(random_state=42).fit(X, y)
initial_depth = dt_full.get_depth()


path = dt_full.cost_complexity_pruning_path(iris.data, iris.target)
alphas = path.ccp_alphas


scores = []
# Ваш код: цикл по alphas, используйте cross_val_score(..., cv=5).mean()

best_alpha = # Альфа с макс. средним скором

In [ ]:
assert len(alphas) > 1
assert all(a >= 0 for a in alphas)
assert 0.0 < best_alpha < 0.1

test_tree = DecisionTreeClassifier(ccp_alpha=best_alpha, random_state=42).fit(X, y)
assert test_tree.get_depth() < initial_depth


assert len(scores) == len(alphas)
assert len(set(np.round(scores, 5))) > 1

## Задание 3: Кастомный GridSearch с кросс-валидацией

`Описание:` Подбор параметров по сетке с использованием `f1-score` вместо стандартного `accuracy`.

`Что нужно сделать:` Используя `GridSearchCV` на данных `load_breast_cancer`, найдите лучшие параметры (`criterion`, `max_depth`, `min_samples_split`). В качестве метрики качества используйте `scoring='f1'`.

In [ ]:
data = load_breast_cancer()
X, y = data.data, data.target

# Настройте сетку параметров и запустите поиск
grid = # GridSearchCV(...)

In [ ]:
assert grid.scoring == 'f1'
assert hasattr(grid, "best_params_")
assert isinstance(grid.best_params_, dict)


assert len(grid.param_grid['max_depth']) > 1
assert len(grid.cv_results_['mean_test_score']) > 1


assert grid.best_score_ > 0.90

## Задание 4: Инвариантность к монотонным преобразованиям

`Описание:` Деревья решений устойчивы не только к масштабированию, но и к любым монотонным преобразованиям признаков (например, логарифмированию).

`Что нужно сделать:` Обучите дерево на данных `load_wine`. Затем примените к признакам функцию $x_{new} = log(x+1)$
 и обучите второе дерево. Сравните топ-3 самых важных признака в обоих случаях.

In [ ]:
wine = load_wine()
X, y = wine.data, wine.target

# Ваш код ниже

dt1 = ...


dt2 = ...

top3_dt1 = np.argsort(dt1.feature_importances_)[-3:]
top3_dt2 = np.argsort(dt2.feature_importances_)[-3:]

In [ ]:
pred1 = dt1.predict(X)
pred2 = dt2.predict(X_log)

assert np.array_equal(pred1, pred2)
assert np.array_equal(np.sort(top3_dt1), np.sort(top3_dt2))

## Задание5: Свое дерево решений (MyDecisionTree)

`Описание:`
Библиотечные функции — это удобно, но важно понимать механику. В этом задании вы реализуете поиск лучшего разбиения (Best Split) на основе `Gini Impurity`. Ваша задача — найти такой порог для признака, который максимально снижает неопределенность в узле.

`Что нужно сделать:`
1. Реализовать расчет `Gini Impurity` для набора меток.
2. Реализовать поиск `Best Split`: перебрать все уникальные значения каждого признака и найти комбинацию (признак, порог), которая дает максимальный `IG`.

In [ ]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split

class MyDecisionTree:
    def __init__(self, max_depth=3):
        self.max_depth = max_depth
        self.tree = None

    def gini(self, y):
        """Вычисляет Gini Impurity для вектора меток y."""
        if len(y) == 0: return 0
        # 1. Посчитайте вероятности каждого класса (p)
        # 2. Верните 1 - sum(p^2)
        # Ваш код здесь
        pass

    def get_best_split(self, X, y):
        """Находит лучший признак и порог для разбиения."""
        best_gain = -1
        split_idx, split_threshold = None, None

        current_gini = self.gini(y)
        n_features = X.shape[1]

        for feat_idx in range(n_features):
            thresholds = np.unique(X[:, feat_idx])
            for threshold in thresholds:
                # Разделите y на левую и правую части по порогу (X <= threshold)
                # Посчитайте Gini Gain: G_parent - (w_left * G_left + w_right * G_right)
                # Ваш код здесь
                pass

        return split_idx, split_threshold

# Загрузка данных для теста
iris = load_iris()
X, y = iris.data[:, :2], iris.target # Берем 2 признака для простоты
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

my_tree = MyDecisionTree()
best_feat, best_threshold = my_tree.get_best_split(X_train, y_train)

In [ ]:
# Проверка gini
test_y = np.array([0,0,1,1])
assert abs(my_tree.gini(test_y) - 0.5) < 1e-6

# Проверка, что split найден
assert best_feat is not None
assert best_threshold is not None

# Проверка корректности разбиения
left_mask = X_train[:, best_feat] <= best_threshold
right_mask = ~left_mask

assert left_mask.sum() > 0
assert right_mask.sum() > 0

# Проверка, что gain > 0
def compute_gain(y, left, right, gini_func):
    w_l = len(left) / len(y)
    w_r = len(right) / len(y)
    return gini_func(y) - (w_l*gini_func(left) + w_r*gini_func(right))

gain = compute_gain(
    y_train,
    y_train[left_mask],
    y_train[right_mask],
    my_tree.gini
)

assert gain > 0

# Анти-чит: проверка на другом датасете
X_rand = np.random.rand(30,2)
y_rand = np.random.randint(0,2,30)

feat, thr = my_tree.get_best_split(X_rand, y_rand)
assert feat is not None